# 🔗 Notebook 05: Association Rules (Aturan Asosiasi)
## Analitika Data Keuangan Sektor Publik

**Tujuan Pembelajaran:**
1. Memahami konsep Support, Confidence, dan Lift
2. Menyiapkan data keuangan untuk analisis asosiasi
3. Menerapkan algoritma Apriori
4. Menginterpretasikan association rules dalam konteks APBD

---
> **Problem:** Temukan **pola hubungan** antara berbagai komponen kinerja anggaran di pemerintah daerah. Misalnya: apakah PEMDA yang merealisasi infrastruktur tinggi juga cenderung merealisasi pendidikan tinggi?

## 📦 1. Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings

try:
    from mlxtend.frequent_patterns import apriori, association_rules
    from mlxtend.preprocessing import TransactionEncoder
    print('mlxtend berhasil dimuat ✅')
except ImportError:
    print('mlxtend belum terinstal. Jalankan: pip install mlxtend')
    print('Melanjutkan dengan implementasi manual...')

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (10, 6)
sns.set_style('whitegrid')

## 📂 2. Persiapan & Transformasi Data

In [ ]:
# Muat dan bersihkan data
df = pd.read_csv('../../Dataset/02-EDA/keuangan_pemda.csv')
df = df[df['Anggaran_APBD'] > 0].copy()
df.dropna(subset=['Anggaran_APBD', 'Realisasi_APBD', 'PAD', 'Dana_Transfer',
                   'Belanja_Pegawai', 'Belanja_Modal'], inplace=True)

# Buat fitur persentase per komponen (simulasi karena kita tidak punya data per sektor)
df['Pct_Realisasi']         = df['Realisasi_APBD'] / df['Anggaran_APBD'] * 100
df['Rasio_Kemandirian']     = df['PAD'] / df['Anggaran_APBD'] * 100
df['Rasio_Transfer']        = df['Dana_Transfer'] / df['Anggaran_APBD'] * 100
df['Rasio_Belanja_Pegawai'] = df['Belanja_Pegawai'] / df['Realisasi_APBD'] * 100
df['Rasio_Belanja_Modal']   = df['Belanja_Modal'] / df['Realisasi_APBD'] * 100

print(f'Data dimuat: {df.shape[0]} PEMDA')
df[['Kode_Pemda', 'Predikat', 'Pct_Realisasi', 'Rasio_Kemandirian',
    'Rasio_Belanja_Pegawai', 'Rasio_Belanja_Modal']].head(8)

In [ ]:
# Diskretisasi — Ubah nilai kontinu menjadi kategori binary
def kategorikan_pct(nilai, kolom):
    """Kategorikan nilai persentase menjadi Tinggi/Rendah berdasarkan median."""
    median = nilai.median()
    return nilai.apply(lambda x: f'{kolom}_TINGGI' if x >= median else f'{kolom}_RENDAH')

df['Kat_Realisasi']        = kategorikan_pct(df['Pct_Realisasi'], 'REALISASI')
df['Kat_Kemandirian']      = kategorikan_pct(df['Rasio_Kemandirian'], 'MANDIRI')
df['Kat_Transfer']         = kategorikan_pct(df['Rasio_Transfer'], 'TRANSFER')
df['Kat_Belanja_Pegawai']  = kategorikan_pct(df['Rasio_Belanja_Pegawai'], 'BEL_PEGAWAI')
df['Kat_Belanja_Modal']    = kategorikan_pct(df['Rasio_Belanja_Modal'], 'BEL_MODAL')
df['Kat_Predikat']         = 'PREDIKAT_' + df['Predikat'].str.upper().str.replace(' ', '_')

kat_cols = ['Kat_Realisasi', 'Kat_Kemandirian', 'Kat_Transfer',
            'Kat_Belanja_Pegawai', 'Kat_Belanja_Modal', 'Kat_Predikat']

print('Contoh setelah diskretisasi:')
df[kat_cols].head(8)

In [ ]:
# Konversi ke format transaksi (list of lists)
transactions = df[kat_cols].values.tolist()

print('Contoh format transaksi:')
for i, t in enumerate(transactions[:5]):
    print(f'  PEMDA {i+1}: {t}')

## 🔍 3. Menerapkan Algoritma Apriori

In [ ]:
# Encode transaksi ke format One-Hot
te = TransactionEncoder()
te_array = te.fit_transform(transactions)
df_trans = pd.DataFrame(te_array, columns=te.columns_)

print(f'Shape matrix transaksi: {df_trans.shape}')
print(f'Jumlah item unik: {df_trans.shape[1]}')
df_trans.head(5)

In [ ]:
# Jalankan Apriori untuk menemukan frequent itemsets
MIN_SUPPORT = 0.25   # Item harus muncul minimal di 25% PEMDA

frequent_itemsets = apriori(df_trans, min_support=MIN_SUPPORT, use_colnames=True)
frequent_itemsets['length'] = frequent_itemsets['itemsets'].apply(len)
frequent_itemsets = frequent_itemsets.sort_values('support', ascending=False)

print(f'Frequent Itemsets ditemukan: {len(frequent_itemsets)}')
print(f'Min Support: {MIN_SUPPORT}')
print()
print('Top 15 Frequent Itemsets:')
print(frequent_itemsets.head(15).to_string(index=False))

In [ ]:
# Generate Association Rules
MIN_CONFIDENCE = 0.6

rules = association_rules(frequent_itemsets, metric='confidence', min_threshold=MIN_CONFIDENCE)
rules = rules.sort_values('lift', ascending=False)

print(f'Association Rules ditemukan: {len(rules)}')
print(f'Min Confidence: {MIN_CONFIDENCE}')
print()

# Format untuk tampilan
rules_display = rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].copy()
rules_display['antecedents'] = rules_display['antecedents'].apply(lambda x: ', '.join(list(x)))
rules_display['consequents'] = rules_display['consequents'].apply(lambda x: ', '.join(list(x)))
rules_display = rules_display.round(4)

print('Top 15 Rules (diurutkan berdasarkan Lift):')
print(rules_display.head(15).to_string(index=False))

## 📊 4. Analisis & Visualisasi Rules

In [ ]:
# Scatter plot: Support vs Confidence (ukuran = Lift)
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Support vs Confidence
scatter = axes[0].scatter(rules['support'], rules['confidence'],
                           c=rules['lift'], cmap='RdYlGn', alpha=0.7,
                           s=rules['lift'] * 30, edgecolors='gray', linewidth=0.5)
plt.colorbar(scatter, ax=axes[0], label='Lift')
axes[0].set_xlabel('Support', fontsize=11)
axes[0].set_ylabel('Confidence', fontsize=11)
axes[0].set_title('Support vs Confidence\n(warna & ukuran = Lift)', fontweight='bold')

# Distribusi Lift
axes[1].hist(rules['lift'], bins=20, color='#3498db', edgecolor='white', linewidth=1.2)
axes[1].axvline(1.0, color='red', linestyle='--', linewidth=1.5, label='Lift = 1 (independen)')
axes[1].set_xlabel('Lift', fontsize=11)
axes[1].set_ylabel('Frekuensi', fontsize=11)
axes[1].set_title('Distribusi Nilai Lift\n(> 1 = hubungan positif)', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()
print(f'\nJumlah rules dengan Lift > 1: {(rules["lift"] > 1).sum()}')

In [ ]:
# Filter rules bermakna (Lift > 1.2 dan Confidence > 0.7)
strong_rules = rules[(rules['lift'] > 1.2) & (rules['confidence'] > 0.7)].copy()
strong_rules = strong_rules.sort_values('lift', ascending=False)

print(f'Strong Rules (Lift > 1.2 & Confidence > 0.7): {len(strong_rules)}')
print()

strong_display = strong_rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].copy()
strong_display['antecedents'] = strong_display['antecedents'].apply(lambda x: ', '.join(list(x)))
strong_display['consequents'] = strong_display['consequents'].apply(lambda x: ', '.join(list(x)))
strong_display = strong_display.round(3)

print('STRONG ASSOCIATION RULES:')
print('=' * 80)
for _, row in strong_display.head(10).iterrows():
    print(f'  {row["antecedents"]:35} → {row["consequents"]:30}')
    print(f'    Support={row["support"]:.3f}  Confidence={row["confidence"]:.3f}  Lift={row["lift"]:.3f}')
    print()

In [ ]:
# Heatmap: Support antar pasangan item
items_of_interest = ['REALISASI_TINGGI', 'REALISASI_RENDAH', 'MANDIRI_TINGGI',
                     'TRANSFER_TINGGI', 'BEL_PEGAWAI_TINGGI', 'BEL_MODAL_TINGGI']

# Filter items yang ada di dataframe
items_of_interest = [i for i in items_of_interest if i in df_trans.columns]

if len(items_of_interest) >= 2:
    support_matrix = pd.DataFrame(index=items_of_interest, columns=items_of_interest, dtype=float)
    for i in items_of_interest:
        for j in items_of_interest:
            support_matrix.loc[i, j] = (df_trans[i] & df_trans[j]).mean()
    
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(support_matrix.astype(float), annot=True, fmt='.2f', cmap='YlOrRd',
                ax=ax, linewidths=0.5, vmin=0, vmax=1)
    ax.set_title('Heatmap Support: Co-occurrence Antar Item\n(diagonal = support tunggal)', fontweight='bold')
    labels = [i.replace('_', ' ').replace('TINGGI', '(T)').replace('RENDAH', '(R)') for i in items_of_interest]
    ax.set_xticklabels(labels, rotation=30, ha='right')
    ax.set_yticklabels(labels, rotation=0)
    plt.tight_layout()
    plt.show()

## 💡 5. Interpretasi Rules

In [ ]:
# Menghitung metrik secara manual untuk pemahaman
print('ILUSTRASI PERHITUNGAN MANUAL')
print('=' * 55)

# Ambil salah satu rule sebagai contoh
if 'REALISASI_TINGGI' in df_trans.columns and 'MANDIRI_TINGGI' in df_trans.columns:
    n_total = len(df_trans)
    n_A = df_trans['REALISASI_TINGGI'].sum()
    n_B = df_trans['MANDIRI_TINGGI'].sum()
    n_AB = (df_trans['REALISASI_TINGGI'] & df_trans['MANDIRI_TINGGI']).sum()

    support_A  = n_A / n_total
    support_B  = n_B / n_total
    support_AB = n_AB / n_total
    confidence = support_AB / support_A
    lift       = confidence / support_B

    print(f'Rule: {{REALISASI_TINGGI}} → {{MANDIRI_TINGGI}}')
    print(f'  N total          = {n_total}')
    print(f'  N(A) REALISASI   = {n_A}')
    print(f'  N(B) MANDIRI     = {n_B}')
    print(f'  N(A∩B)           = {n_AB}')
    print(f'')
    print(f'  Support(A∩B)     = {n_AB}/{n_total} = {support_AB:.4f} ({support_AB*100:.1f}%)')
    print(f'  Confidence(A→B)  = {support_AB:.4f}/{support_A:.4f} = {confidence:.4f} ({confidence*100:.1f}%)')
    print(f'  Lift(A→B)        = {confidence:.4f}/{support_B:.4f} = {lift:.4f}')
    print()
    if lift > 1:
        print(f'  ✅ Lift > 1 → Ada korelasi POSITIF antara REALISASI_TINGGI dan MANDIRI_TINGGI')
    else:
        print(f'  ⚠️ Lift ≤ 1 → Tidak ada korelasi positif yang bermakna')

## 📝 6. Latihan

1. Ubah `MIN_SUPPORT` menjadi 0.15 dan 0.35. Bagaimana jumlah rules berubah?
2. Filter rules yang memiliki `PREDIKAT_KURANG` sebagai consequent. Apa antecedent-nya?
3. Cari rules dengan `PREDIKAT_SANGAT_BAIK` — apa yang biasanya muncul bersamaan?
4. **Diskusi:** Apakah rules yang ditemukan masuk akal secara fiskal? Jelaskan!
5. Tambahkan fitur `Kat_Anggaran_Besar` (Anggaran > median). Apakah ada rules baru yang menarik?

In [ ]:
# ✏️ Ruang Eksplorasi — Filter rules berdasarkan consequent
target_consequent = 'PREDIKAT_KURANG'

if target_consequent in rules_display['consequents'].values:
    filtered = rules_display[rules_display['consequents'].str.contains(target_consequent)]
    print(f'Rules dengan consequent {target_consequent}:')
    print(filtered.sort_values('lift', ascending=False).head(10).to_string(index=False))
else:
    print(f'Tidak ditemukan rules dengan consequent: {target_consequent}')
    print('Rules yang tersedia:')
    print(rules_display['consequents'].value_counts().head(10))